# Combined SI Plots (Readout / Raw / Gate-Corrected)

This notebook generates combined panels in the same layout as the provided Fig. 7 and Fig. 8 examples.


## Load Data

Loads singlet `alpha, P, FI` CSV files for three styles: `original` (readout-fidelity-corrected), `raw`, and `corrected` (gate-fidelity-corrected).


In [ ]:
import os
from pathlib import Path

# Ensure relative paths work regardless of where the notebook is launched from.
_repo_root = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "data").exists():
        _repo_root = p
        break
if _repo_root is not None:
    os.chdir(_repo_root)
print("Working directory:", Path.cwd())

import os
from pathlib import Path
import numpy as np

base_dir_candidates = [
    Path('data_analysis'),
    Path('../Data_analysis'),
    Path.cwd() / 'data_analysis',
    Path.cwd().parent / 'data_analysis',
]
base_dir = None
for p in base_dir_candidates:
    if p.exists():
        base_dir = p
        break

if base_dir is None:
    raise FileNotFoundError('Could not locate Data_analysis directory.')

date_tag = '1218'
styles = ['original', 'raw', 'corrected']
axes_order = ['Ux', 'Uy', 'Uz']

def load_alpha_P_FI(axis_name, style):
    fname = f'singlet_{axis_name}_alpha_P_FI_{date_tag}_{style}.csv'
    fpath = base_dir / fname
    if not fpath.exists():
        matches = sorted(base_dir.glob(f'singlet_{axis_name}_alpha_P_FI_*_{style}.csv'))
        if not matches:
            raise FileNotFoundError(f'Missing file for singlet_{axis_name}, style={style}')
        fpath = matches[-1]

    arr = np.loadtxt(fpath, delimiter=',', skiprows=1)
    if arr.ndim == 1:
        arr = arr[None, :]

    return {
        'alpha': arr[:, 0],
        'P': arr[:, 1],
        'FI': arr[:, 2],
        'path': str(fpath),
    }

data = {}
for style in styles:
    data[style] = {}
    for ax in axes_order:
        data[style][ax] = load_alpha_P_FI(ax, style)

for style in styles:
    print(f'[{style}]')
    for ax in axes_order:
        print('  ', data[style][ax]['path'])


## Prepare Common Alpha Grids

For each style, curves are clipped to `alpha <= 2π`, sorted, interpolated to one shared overlap grid, and downsampled for plotting.


In [ ]:
import numpy as np

def prep_alpha_y(alpha, y, alpha_max=2*np.pi):
    alpha = np.asarray(alpha, float)
    y = np.asarray(y, float)

    m = np.isfinite(alpha) & np.isfinite(y) & (alpha <= alpha_max)
    alpha = alpha[m]
    y = y[m]

    order = np.argsort(alpha)
    alpha = alpha[order]
    y = y[order]
    return alpha, y

def build_style_curves(style_entry, downsample_frac=0.7):
    # Prepare per-axis arrays
    alphaP = {}
    Pvals = {}
    alphaFI = {}
    FIvals = {}

    for ax in ['Ux', 'Uy', 'Uz']:
        aP, p = prep_alpha_y(style_entry[ax]['alpha'], style_entry[ax]['P'])
        aF, fi = prep_alpha_y(style_entry[ax]['alpha'], style_entry[ax]['FI'])
        alphaP[ax], Pvals[ax] = aP, p
        alphaFI[ax], FIvals[ax] = aF, fi

    # Common overlap grid across axes (using alpha ranges from P arrays)
    lo = max(np.min(alphaP['Ux']), np.min(alphaP['Uy']), np.min(alphaP['Uz']))
    hi = min(np.max(alphaP['Ux']), np.max(alphaP['Uy']), np.max(alphaP['Uz']))
    if not (hi > lo):
        raise ValueError('No overlap in alpha across Ux/Uy/Uz for this style.')

    N = min(len(alphaP['Ux']), len(alphaP['Uy']), len(alphaP['Uz']))
    alpha_common = np.linspace(lo, hi, N)

    P_Ux = np.interp(alpha_common, alphaP['Ux'], Pvals['Ux'])
    P_Uy = np.interp(alpha_common, alphaP['Uy'], Pvals['Uy'])
    P_Uz = np.interp(alpha_common, alphaP['Uz'], Pvals['Uz'])

    FI_Ux = np.interp(alpha_common, alphaFI['Ux'], FIvals['Ux'])
    FI_Uy = np.interp(alpha_common, alphaFI['Uy'], FIvals['Uy'])
    FI_Uz = np.interp(alpha_common, alphaFI['Uz'], FIvals['Uz'])

    FI_mean = (FI_Ux + FI_Uy + FI_Uz) / 3.0

    # Downsample for display
    n = len(alpha_common)
    m = max(2, int(round(downsample_frac * n)))
    idx = np.linspace(0, n - 1, m).astype(int)

    return {
        'alpha': alpha_common[idx],
        'P_Ux': P_Ux[idx],
        'P_Uy': P_Uy[idx],
        'P_Uz': P_Uz[idx],
        'FI_Ux': FI_Ux[idx],
        'FI_Uy': FI_Uy[idx],
        'FI_Uz': FI_Uz[idx],
        'FI_mean': FI_mean[idx],
    }

curves = {style: build_style_curves(data[style], downsample_frac=0.7) for style in styles}

# Average FI horizontal levels per style
fi_levels = {}
for style in styles:
    fi_mean = curves[style]['FI_mean']
    if style == 'corrected':
        # Gate-corrected case: exclude singular points when computing average level
        mask = np.isfinite(fi_mean) & (fi_mean <= 4)
        fi_levels[style] = float(np.mean(fi_mean[mask]))
    else:
        fi_levels[style] = float(np.mean(fi_mean[np.isfinite(fi_mean)]))

print('Average FI levels:', fi_levels)


## Plot Style


In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

lw = 1
mpl.rcParams['axes.linewidth'] = lw
mpl.rcParams['lines.linewidth'] = lw
mpl.rcParams['xtick.major.width'] = lw
mpl.rcParams['ytick.major.width'] = lw
mpl.rcParams['xtick.major.size'] = 2
mpl.rcParams['ytick.major.size'] = 2
mpl.rcParams['font.size'] = 8
mpl.rcParams['axes.labelsize'] = 8
mpl.rcParams['axes.titlesize'] = 8
mpl.rcParams['legend.fontsize'] = 7
mpl.rcParams['xtick.labelsize'] = 7
mpl.rcParams['ytick.labelsize'] = 7

style_x = dict(color='#6D00C9', marker='s', linestyle='-', markersize=5.0, label=r'$\hat{x}$')
style_y = dict(color='#F9AE2A', marker='^', linestyle='-', markersize=5.0, label=r'$\hat{y}$')
style_z = dict(color='#777777', marker='o', linestyle='-', markersize=5.0, label=r'$\hat{z}$')

xticks = [0, np.pi, 2*np.pi]
xticklabels = [r'$0$', r'$\pi$', r'$2\pi$']

style_mean = dict(color='blue', marker='o', linestyle='-', markersize=5.0, label=r'FI from $|\Psi^-\rangle$')
style_hline = dict(color='red', linestyle='--', linewidth=1.2, label=r'Avg. FI from $|\Psi^-\rangle$')


## Fig. 7 Format: Probability and FI (2x3)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

style_order = ['original', 'raw', 'corrected']
panel_letters = ['a', 'b', 'c']

# Approximate size/aspect of attached figure
fig, axes = plt.subplots(2, 3, figsize=(9.0, 4.15), dpi=300, sharex='col')

for j, style in enumerate(style_order):
    c = curves[style]

    # Top row: Probability
    ax = axes[0, j]
    ax.plot(c['alpha'], c['P_Ux'], **style_x)
    ax.plot(c['alpha'], c['P_Uy'], **style_y)
    ax.plot(c['alpha'], c['P_Uz'], **style_z)
    ax.set_xlim(-0.2, 2*np.pi + 0.2)
    ax.set_ylim(-0.1, 1.05)
    ax.set_xticks(xticks)
    ax.set_yticks([0, 0.5, 1.0])
    ax.tick_params(axis='both', which='both', direction='in')
    ax.tick_params(axis='x', labelbottom=False, length=4, width=lw, direction='in')

    # Bottom row: FI
    ax2 = axes[1, j]
    ax2.plot(c['alpha'], c['FI_Ux'], **style_x)
    ax2.plot(c['alpha'], c['FI_Uy'], **style_y)
    ax2.plot(c['alpha'], c['FI_Uz'], **style_z)
    ax2.set_xlim(-0.2, 2*np.pi + 0.2)
    ax2.set_ylim(-0.2, 4.2)
    ax2.set_xticks(xticks)
    ax2.set_xticklabels(xticklabels)
    ax2.set_yticks([0, 2, 4])
    ax2.tick_params(axis='both', which='both', direction='in')
    ax2.tick_params(axis='x', length=4, width=lw, direction='in')
    ax2.set_xlabel(r'$\alpha$ (radians)')

    # Panel letters
    axes[0, j].text(-0.06, 1.06, panel_letters[j], transform=axes[0, j].transAxes,
                    fontsize=12, fontweight='bold', va='bottom', ha='left')

# Shared y-labels (left column only)
axes[0, 0].set_ylabel(r'$P(|\Psi^-\rangle)$')
axes[1, 0].set_ylabel('FI')

# Hide repeated y tick labels for inner/right columns
for j in [1, 2]:
    axes[0, j].set_yticklabels([])
    axes[1, j].set_yticklabels([])

# Legend only on top-right panel, outside axis
axes[0, 2].legend(frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5),
                  handlelength=1.5, handletextpad=0.4, borderaxespad=0.0)

fig.subplots_adjust(left=0.06, right=0.90, top=0.93, bottom=0.11, wspace=0.08, hspace=0.14)

outpath = os.path.join('si_figures_pub', 'SI_fig7_combined.pdf')
os.makedirs(os.path.dirname(outpath), exist_ok=True)
fig.savefig(outpath, bbox_inches='tight')
plt.show()
print('Saved:', outpath)


FIG. 7. **Singlet-state probabilities and Fisher information under rotations about the** $\hat{x}$, $\hat{y}$, **and** $\hat{z}$ **axes.** Top panels (a)-(c) show the measured probability $P(|\Psi^-\rangle)$ as a function of the rotation angle $\alpha$ for the *readout-fidelity-corrected*, *raw*, and *gate-fidelity-corrected* data sets, respectively. Bottom panels (a)-(c) show the corresponding classical Fisher information (FI) extracted from the probability as a function of $\alpha$, for the three cases.


## Fig. 8 Format: Average FI (1x3)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

style_order = ['original', 'raw', 'corrected']
panel_letters = ['a', 'b', 'c']

fig, axes = plt.subplots(1, 3, figsize=(9.0, 2.3), dpi=300, sharey=True)

for j, style in enumerate(style_order):
    ax = axes[j]
    c = curves[style]
    level = fi_levels[style]

    ax.plot(c['alpha'], c['FI_mean'], **style_mean)
    ax.axhline(level, **style_hline)

    ax.set_xlim(-0.2, 2*np.pi + 0.2)
    ax.set_ylim(-0.2, 4.2)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_yticks([0, 2, 4])
    ax.set_xlabel(r'$\alpha$ (radians)')
    ax.tick_params(axis='both', which='both', direction='in')
    ax.tick_params(axis='x', length=4, width=lw, direction='in')

    ax.text(-0.06, 1.06, panel_letters[j], transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='bottom', ha='left')

axes[0].set_ylabel('FI')

# Legend only on panel c, outside axis
axes[2].legend(frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5),
               handlelength=1.5, handletextpad=0.4, borderaxespad=0.0)

fig.subplots_adjust(left=0.06, right=0.84, top=0.90, bottom=0.25, wspace=0.08)

outpath = os.path.join('si_figures_pub', 'SI_fig8_average_FI.pdf')
os.makedirs(os.path.dirname(outpath), exist_ok=True)
fig.savefig(outpath, bbox_inches='tight')
plt.show()
print('Saved:', outpath)
print('Average FI levels:', fi_levels)


FIG. 8. **Average Fisher information under rotations about the** $\hat{x}$, $\hat{y}$, **and** $\hat{z}$ **axes.** Panels (a)-(c) show the average Fisher information as a function of the rotation angle $\alpha$ for the *readout-fidelity-corrected*, *raw*, and *gate-fidelity-corrected* data sets and the horizontal dashed lines are the average Fisher information over all values of $\alpha$. The *gate-fidelity-corrected* data set has singularities when calculating the FI which are discarded when averaging FI over $\alpha$.


## Fig. 3 Main-Text Format

This section builds a composite figure with panels `a`-`g` in the same layout and style family as Fig. 3 in the main text.


In [ ]:
import numpy as np
from pathlib import Path

# Data loader for a full set of Ux/Uy/Uz curves for one system/style
def load_system_curves(system, style='original', date_tag='1218'):
    out = {}
    for ax in ['Ux', 'Uy', 'Uz']:
        f = base_dir / f'{system}_{ax}_alpha_P_FI_{date_tag}_{style}.csv'
        if not f.exists():
            raise FileNotFoundError(f'Missing file: {f}')
        arr = np.loadtxt(f, delimiter=',', skiprows=1)
        out[ax] = {
            'alpha': arr[:, 0],
            'P': arr[:, 1],
            'FI': arr[:, 2],
        }
    return out

def prep_alpha_y(alpha, y, alpha_max=2*np.pi):
    alpha = np.asarray(alpha, float)
    y = np.asarray(y, float)
    m = np.isfinite(alpha) & np.isfinite(y) & (alpha <= alpha_max)
    alpha = alpha[m]
    y = y[m]
    order = np.argsort(alpha)
    return alpha[order], y[order]

def build_common_curves(system_data, downsample_frac=0.7):
    aP = {}
    P = {}
    aF = {}
    FI = {}

    for ax in ['Ux', 'Uy', 'Uz']:
        aP[ax], P[ax] = prep_alpha_y(system_data[ax]['alpha'], system_data[ax]['P'])
        aF[ax], FI[ax] = prep_alpha_y(system_data[ax]['alpha'], system_data[ax]['FI'])

    lo = max(np.min(aP['Ux']), np.min(aP['Uy']), np.min(aP['Uz']))
    hi = min(np.max(aP['Ux']), np.max(aP['Uy']), np.max(aP['Uz']))
    N = min(len(aP['Ux']), len(aP['Uy']), len(aP['Uz']))
    alpha = np.linspace(lo, hi, N)

    P_Ux = np.interp(alpha, aP['Ux'], P['Ux'])
    P_Uy = np.interp(alpha, aP['Uy'], P['Uy'])
    P_Uz = np.interp(alpha, aP['Uz'], P['Uz'])

    FI_Ux = np.interp(alpha, aF['Ux'], FI['Ux'])
    FI_Uy = np.interp(alpha, aF['Uy'], FI['Uy'])
    FI_Uz = np.interp(alpha, aF['Uz'], FI['Uz'])

    FI_mean = (FI_Ux + FI_Uy + FI_Uz) / 3.0

    m = max(2, int(round(downsample_frac * len(alpha))))
    idx = np.linspace(0, len(alpha) - 1, m).astype(int)

    return {
        'alpha': alpha[idx],
        'P_Ux': P_Ux[idx],
        'P_Uy': P_Uy[idx],
        'P_Uz': P_Uz[idx],
        'FI_Ux': FI_Ux[idx],
        'FI_Uy': FI_Uy[idx],
        'FI_Uz': FI_Uz[idx],
        'FI_mean': FI_mean[idx],
    }

# Main-text style data choice: use the readout-fidelity-corrected ('original') set
singlet_main = build_common_curves(load_system_curves('singlet', style='original'))
q1_main = build_common_curves(load_system_curves('Q1', style='original'))
q2_main = build_common_curves(load_system_curves('Q2', style='original'))

# Panel g: compare singlet mean FI vs entanglement-free FI (= mean(Q1) + mean(Q2))
alpha_g_lo = max(np.min(singlet_main['alpha']), np.min(q1_main['alpha']), np.min(q2_main['alpha']))
alpha_g_hi = min(np.max(singlet_main['alpha']), np.max(q1_main['alpha']), np.max(q2_main['alpha']))
alpha_g = np.linspace(alpha_g_lo, alpha_g_hi, min(len(singlet_main['alpha']), len(q1_main['alpha']), len(q2_main['alpha'])))

FI_metrology = np.interp(alpha_g, singlet_main['alpha'], singlet_main['FI_mean'])
FI_entfree = np.interp(alpha_g, q1_main['alpha'], q1_main['FI_mean']) + np.interp(alpha_g, q2_main['alpha'], q2_main['FI_mean'])

FI_metrology_avg = float(np.mean(FI_metrology[np.isfinite(FI_metrology)]))
FI_entfree_avg = float(np.mean(FI_entfree[np.isfinite(FI_entfree)]))

print('Average FI (metrology):', FI_metrology_avg)
print('Average FI (entanglement-free):', FI_entfree_avg)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Color/marker palette to match the reference figure style
style_x_main = dict(color='#1f4bd8', marker='s', linestyle='-', markersize=4.5, linewidth=1.0, label=r'$\hat{x}$')
style_y_main = dict(color='#f2a007', marker='^', linestyle='-', markersize=4.5, linewidth=1.0, label=r'$\hat{y}$')
style_z_main = dict(color='#7a7a7a', marker='o', linestyle='-', markersize=4.5, linewidth=1.0, label=r'$\hat{z}$')

xt = [0, np.pi, 2*np.pi]
xtl = [r'$0$', r'$\pi$', r'$2\pi$']

# Layout without panels a/d (keep b,c,e,f,g format)
fig = plt.figure(figsize=(10.2, 8.1), dpi=300)
gs = fig.add_gridspec(
    2, 3,
    width_ratios=[1.55, 1.55, 1.45],
    height_ratios=[1.0, 2.0],
    left=0.08, right=0.985, top=0.94, bottom=0.22,
    wspace=0.42, hspace=0.35
)

ax_b = fig.add_subplot(gs[0, 0])
ax_c = fig.add_subplot(gs[0, 1])
ax_g = fig.add_subplot(gs[0, 2])

gs_e = gs[1, 0].subgridspec(2, 1, hspace=0.18)
ax_e_top = fig.add_subplot(gs_e[0, 0])
ax_e_bot = fig.add_subplot(gs_e[1, 0])

gs_f = gs[1, 1].subgridspec(2, 1, hspace=0.18)
ax_f_top = fig.add_subplot(gs_f[0, 0])
ax_f_bot = fig.add_subplot(gs_f[1, 0])

ax_leg = fig.add_subplot(gs[1, 2])
ax_leg.axis('off')

# ---------- Panel b: singlet probability ----------
ax_b.plot(singlet_main['alpha'], singlet_main['P_Ux'], **style_x_main)
ax_b.plot(singlet_main['alpha'], singlet_main['P_Uy'], **style_y_main)
ax_b.plot(singlet_main['alpha'], singlet_main['P_Uz'], **style_z_main)
ax_b.set_xlim(-0.2, 2*np.pi + 0.2)
ax_b.set_ylim(-0.05, 1.05)
ax_b.set_xticks(xt)
ax_b.set_xticklabels(xtl)
ax_b.set_yticks([0, 0.5, 1.0])
ax_b.set_xlabel(r'$\alpha$ (radians)')
ax_b.set_ylabel(r'$P(|\Psi^-\rangle)$')
ax_b.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_b.text(-0.35, 1.04, 'b', transform=ax_b.transAxes, fontsize=11, fontweight='bold')

ax_b.legend(frameon=False, loc='center left', bbox_to_anchor=(1.03, 0.62),
            handlelength=1.2, handletextpad=0.35, borderaxespad=0.0)

# ---------- Panel c: singlet FI ----------
ax_c.plot(singlet_main['alpha'], singlet_main['FI_Ux'], **style_x_main)
ax_c.plot(singlet_main['alpha'], singlet_main['FI_Uy'], **style_y_main)
ax_c.plot(singlet_main['alpha'], singlet_main['FI_Uz'], **style_z_main)
ax_c.set_xlim(-0.2, 2*np.pi + 0.2)
ax_c.set_ylim(-0.05, 4.2)
ax_c.set_xticks(xt)
ax_c.set_xticklabels(xtl)
ax_c.set_yticks([0, 2, 4])
ax_c.set_xlabel(r'$\alpha$ (radians)')
ax_c.set_ylabel('FI')
ax_c.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_c.text(-0.30, 1.04, 'c', transform=ax_c.transAxes, fontsize=11, fontweight='bold')

# ---------- Panel e: entanglement-free probabilities ----------
ax_e_top.plot(q2_main['alpha'], q2_main['P_Ux'], **style_x_main)
ax_e_top.plot(q2_main['alpha'], q2_main['P_Uy'], **style_y_main)
ax_e_top.plot(q2_main['alpha'], q2_main['P_Uz'], **style_z_main)
ax_e_top.set_xlim(-0.2, 2*np.pi + 0.2)
ax_e_top.set_ylim(-0.05, 1.05)
ax_e_top.set_xticks(xt)
ax_e_top.set_xticklabels([])
ax_e_top.set_yticks([0, 0.5, 1.0])
ax_e_top.set_ylabel(r'$P(|z+\rangle)$')
ax_e_top.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_e_top.text(1.02, 0.85, r'$|z+\rangle$', transform=ax_e_top.transAxes, fontsize=10, ha='left', va='center')

ax_e_bot.plot(q1_main['alpha'], q1_main['P_Ux'], **style_x_main)
ax_e_bot.plot(q1_main['alpha'], q1_main['P_Uy'], **style_y_main)
ax_e_bot.plot(q1_main['alpha'], q1_main['P_Uz'], **style_z_main)
ax_e_bot.set_xlim(-0.2, 2*np.pi + 0.2)
ax_e_bot.set_ylim(-0.05, 1.05)
ax_e_bot.set_xticks(xt)
ax_e_bot.set_xticklabels(xtl)
ax_e_bot.set_yticks([0, 0.5, 1.0])
ax_e_bot.set_xlabel(r'$\alpha$ (radians)')
ax_e_bot.set_ylabel(r'$P(|x+\rangle)$')
ax_e_bot.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_e_bot.text(1.02, 0.85, r'$|x+\rangle$', transform=ax_e_bot.transAxes, fontsize=10, ha='left', va='center')
ax_e_top.text(-0.30, 1.04, 'e', transform=ax_e_top.transAxes, fontsize=11, fontweight='bold')

# ---------- Panel f: entanglement-free FI ----------
ax_f_top.plot(q2_main['alpha'], q2_main['FI_Ux'], **style_x_main)
ax_f_top.plot(q2_main['alpha'], q2_main['FI_Uy'], **style_y_main)
ax_f_top.plot(q2_main['alpha'], q2_main['FI_Uz'], **style_z_main)
ax_f_top.set_xlim(-0.2, 2*np.pi + 0.2)
ax_f_top.set_ylim(-0.02, 1.05)
ax_f_top.set_xticks(xt)
ax_f_top.set_xticklabels([])
ax_f_top.set_yticks([0, 0.5, 1.0])
ax_f_top.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)

ax_f_bot.plot(q1_main['alpha'], q1_main['FI_Ux'], **style_x_main)
ax_f_bot.plot(q1_main['alpha'], q1_main['FI_Uy'], **style_y_main)
ax_f_bot.plot(q1_main['alpha'], q1_main['FI_Uz'], **style_z_main)
ax_f_bot.set_xlim(-0.2, 2*np.pi + 0.2)
ax_f_bot.set_ylim(-0.02, 1.05)
ax_f_bot.set_xticks(xt)
ax_f_bot.set_xticklabels(xtl)
ax_f_bot.set_yticks([0, 0.5, 1.0])
ax_f_bot.set_xlabel(r'$\alpha$ (radians)')
ax_f_bot.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_f_top.text(-0.28, 1.04, 'f', transform=ax_f_top.transAxes, fontsize=11, fontweight='bold')

# ---------- Panel g: FI comparison and averages ----------
ax_g.plot(alpha_g, FI_metrology, color='black', marker='o', markersize=3.6, linewidth=1.0)
ax_g.plot(alpha_g, FI_entfree, color='#45b2e8', marker='s', markersize=3.6, linewidth=1.0)
ax_g.axhline(FI_metrology_avg, color='red', linestyle='--', linewidth=1.0)
ax_g.axhline(FI_entfree_avg, color='red', linestyle='-.', linewidth=1.0)
ax_g.set_xlim(-0.2, 2*np.pi + 0.2)
ax_g.set_ylim(-0.05, 4.2)
ax_g.set_xticks(xt)
ax_g.set_xticklabels(xtl)
ax_g.set_yticks([0, 2, 4])
ax_g.set_xlabel(r'$\alpha$ (radians)')
ax_g.set_ylabel(r'FI from $v_{st}=2$')
ax_g.tick_params(axis='both', which='both', direction='in', top=True, right=True, length=3)
ax_g.text(-0.26, 1.04, 'g', transform=ax_g.transAxes, fontsize=11, fontweight='bold')

legend_handles = [
    Line2D([0], [0], color='black', marker='o', linestyle='None', markersize=4, label=r'FI from $|\Psi^-\rangle$'),
    Line2D([0], [0], color='#45b2e8', marker='s', linestyle='None', markersize=4,
           label=r'FI from $|x+\rangle_q \otimes |z+\rangle_{\bar{q}}$'),
    Line2D([0], [0], color='red', linestyle='--', linewidth=1.2, label=r'Avg. FI from $|\Psi^-\rangle$'),
    Line2D([0], [0], color='red', linestyle='-.', linewidth=1.2,
           label=r'Avg. FI from $|x+\rangle_q \otimes |z+\rangle_{\bar{q}}$'),
]
ax_leg.legend(handles=legend_handles, frameon=False, loc='upper left',
              handlelength=1.4, handletextpad=0.35, borderaxespad=0.0)

outpath = os.path.join('si_figures_pub', 'SI_fig3_maintext_format.pdf')
os.makedirs(os.path.dirname(outpath), exist_ok=True)
fig.savefig(outpath, bbox_inches='tight')

outpath_png = os.path.join('si_figures_pub', 'SI_fig3_maintext_format.png')
fig.savefig(outpath_png, dpi=300, bbox_inches='tight')

plt.show()
print('Saved:', outpath)
print('Saved:', outpath_png)


FIG. 3 (replotted format, without circuit panels a/d). Panels **(b)** and **(c)** show the singlet probability $P(|\Psi^-\rangle)$ and FI versus $\alpha$, respectively, for rotations about $\hat{x},\hat{y},\hat{z}$. Panels **(e)** and **(f)** show the corresponding entanglement-free probabilities and FI. Panel **(g)** compares the average FI inferred from singlet metrology and entanglement-free metrology, with horizontal red lines indicating averages over all plotted $\alpha$.
